In [1]:
import pandas as pd

# df = pd.read_parquet('.intermediate/sector_info.parquet')
# df = pd.read_parquet('.cache/ohlcv_master_cache.parquet')
df = pd.read_parquet('output/stock_screener.parquet')
# df = pd.read_parquet('.intermediate/close.parquet')


In [2]:
pd.set_option('display.max_columns', None)

cols = ['ticker', 'market_cap', 'industry', 'rs_line_ma_stack', 'rs_percentile', 'rs_3m_pct', 'rs_1m_pct', 'atr_multiple']

ticker = 'CF'

df[df['ticker'] == ticker][cols]

,ticker,market_cap,industry,rs_line_ma_stack,rs_percentile,rs_3m_pct,rs_1m_pct,atr_multiple
563,CF,2.052145e+10,Agricultural Inputs,정배열,89.626784,85.254299,89.042799,3.755043


In [11]:
df[df['industry'] == 'Copper'][cols]

,ticker,market_cap,industry,rs_line_ma_stack,rs_percentile,rs_3m_pct,rs_1m_pct,atr_multiple
252,TGB,3.363754e+09,Copper,정배열,95.363385,92.272309,92.057826,4.467399
258,ERO,4.111610e+09,Copper,정배열,95.252990,93.983441,98.126004,7.337500
361,HBM,1.335099e+10,Copper,정배열,93.339466,83.569457,94.449402,5.249710
493,SCCO,1.802156e+11,Copper,정배열,90.892364,82.907084,88.434767,4.690339
563,FCX,1.073240e+11,Copper,정배열,89.586017,84.802208,88.524005,4.932072
588,TECK,3.394952e+10,Copper,정배열,89.126035,68.794848,84.026414,3.965561
4076,IE,1.783946e+09,Copper,정배열,23.882245,21.803128,91.022666,2.616529


In [3]:
import plotly.express as px
import pandas as pd

df = pd.read_parquet('output/stock_screener.parquet')

# 1. 유의미한 분석을 위해 종목 수가 일정 기준(예: 5개 이상)인 Industry만 필터링
industry_counts = df['industry'].value_counts()
valid_industries = industry_counts[industry_counts >= 5].index
df_filtered = df[df['industry'].isin(valid_industries)].copy()

# 2. RS 중앙값 기준으로 정렬 순서 계산
order = df_filtered.groupby('industry')['rs_percentile'].median().sort_values(ascending=False).index[:30] # 상위 30개
df_top = df_filtered[df_filtered['industry'].isin(order)]

# 3. Box Plot + 개별 종목 점(Scatter) 시각화
fig = px.box(
    df_top,
    x='industry',
    y='rs_percentile',
    category_orders={'industry': list(order)},
    points='all',  # 모든 종목을 점으로 함께 표시
    hover_data=['ticker', 'market_cap', 'rs_score', 'theme'],
    title='Top Industries by RS Percentile Distribution (Hover for Tickers)',
    labels={'rs_percentile': 'RS Percentile', 'industry': 'Industry'},
    template='plotly_dark',
    height=700
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()
